# День 2 — Получение эмбеддингов (hidden states)

**Цель дня:** научиться извлекать скрытые состояния (hidden states) из трансформерной модели.

Используем код из Дня 1 (`tokenize_texts` из [`tokenization_utils.py`](tokenization_utils.py)). Переиспользуемые функции этого дня — в [`embeddings_utils.py`](embeddings_utils.py). Понадобятся в День 3 (визуализация) и День 4 (классификация).

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Задача 1: Загрузка модели

In [2]:
model = AutoModel.from_pretrained(model_name)
model.eval()  # режим оценки: отключаем dropout и т.п.

print(model)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8779.29it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

В архитектуре видно: слой эмбеддингов (`embeddings`) + стек трансформерных блоков (`transformer` с несколькими `TransformerBlock`). У DistilBERT их 6, hidden_size = 768.

## Задача 2: Получение hidden states для одного текста

In [3]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**tokens)

print(type(outputs))
print(outputs.last_hidden_state.shape)  # [batch_size, sequence_length, hidden_size]

<class 'transformers.modeling_outputs.BaseModelOutput'>
torch.Size([1, 8, 768])


In [4]:
# CLS-токен — первый в последовательности (позиция 0)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f'CLS embedding shape: {cls_embedding.shape}')
print(f'CLS embedding: {cls_embedding[0][:5]}...')  # первые 5 значений

CLS embedding shape: torch.Size([1, 768])
CLS embedding: tensor([ 0.0682, -0.0685,  0.1918,  0.0139, -0.0544])...


`last_hidden_state` имеет форму `[batch, seq_len, hidden]`. Срез `[:, 0, :]` берёт вектор CLS-токена — его часто используют как эмбеддинг всего предложения.

## Задача 3: Функция для получения эмбеддингов

Локальное определение для наглядности (ниже покажем и импорт из модуля).

In [5]:
import numpy as np


def get_embeddings(texts, tokenizer, model, batch_size=32):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # Токенизируем (как в Дне 1)
        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        # Получаем hidden states
        with torch.no_grad():
            outputs = model(**tokens)

        # Извлекаем CLS-токены
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

## Задача 4: Тестирование на нескольких текстах

In [6]:
texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long.",
]

embeddings = get_embeddings(texts, tokenizer, model)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Ожидается: (4, 768) для DistilBERT')

Embeddings shape: (4, 768)
Ожидается: (4, 768) для DistilBERT


In [7]:
# То же самое через переиспользуемый модуль
from embeddings_utils import load_model, get_embeddings as get_embeddings_util

emb_util = get_embeddings_util(texts, tokenizer, model)
print(f'Shape из модуля: {emb_util.shape}')

Shape из модуля: (4, 768)


## Задача 5: Сходство текстов

In [8]:
from sklearn.metrics.pairwise import cosine_similarity


def similarity(text1, text2, tokenizer, model):
    emb = get_embeddings([text1, text2], tokenizer, model)
    sim = cosine_similarity(emb[0:1], emb[1:2])[0][0]
    return sim


sim1 = similarity("Great movie!", "Amazing film!", tokenizer, model)
sim2 = similarity("Great movie!", "Terrible film!", tokenizer, model)

print(f'Сходство похожих: {sim1:.3f}')
print(f'Сходство разных: {sim2:.3f}')

Сходство похожих: 0.995
Сходство разных: 0.984


Похожие по смыслу тексты дают более высокое косинусное сходство, чем противоположные по тональности.

> Замечание: у «сырого» DistilBERT (без дообучения) CLS-эмбеддинги не идеально разделяют смысл — значения сходства обычно высокие для всех пар. Полноценная оценка тональности появится в День 4 при классификации.

## Чекпоинт

К концу дня есть:
- ✅ Загруженная модель `AutoModel` (в режиме eval)
- ✅ Функция `get_embeddings` для эмбеддингов батчами
- ✅ Понимание hidden states и CLS-токена (`[batch, seq_len, hidden]`)
- ✅ Функция `similarity` для косинусного сходства

Переиспользуемый код — в [`embeddings_utils.py`](embeddings_utils.py).